In [ ]:
import numpy as np
import pandas as pd
print('numpy', np.__version__, 'pandas', pd.__version__)

In [ ]:
# Cell A: generate a 80M-row DataFrame with random keys + 4 numeric columns.
# Sized so this cell alone takes several seconds and produces ~500MB of
# data the downstream cells will operate on.
rng = np.random.default_rng(42)
N = 80_000_000
df = pd.DataFrame({
    'group_a': rng.integers(0, 1000, N, dtype=np.int32),
    'group_b': rng.integers(0, 50, N, dtype=np.int32),
    'group_c': rng.integers(0, 10, N, dtype=np.int32),
    'value_1': rng.standard_normal(N).astype(np.float32),
    'value_2': rng.exponential(1.0, N).astype(np.float32),
    'value_3': rng.uniform(-1, 1, N).astype(np.float32),
    'value_4': rng.lognormal(0, 1, N).astype(np.float32),
})
print(f'df: {df.shape}, memory: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB')

In [ ]:
# Cell B: three multi-key groupby aggregations. Each takes seconds at 80M rows.
agg_ab = df.groupby(['group_a', 'group_b']).agg(
    v1_mean=('value_1', 'mean'),
    v1_std=('value_1', 'std'),
    v2_sum=('value_2', 'sum'),
    v3_max=('value_3', 'max'),
    v4_min=('value_4', 'min'),
    n=('value_1', 'count'),
).reset_index()
print(f'agg_ab: {agg_ab.shape}')

In [ ]:
# Cell C: groupby across all three keys; produces a wider/longer result.
agg_abc = df.groupby(['group_a', 'group_b', 'group_c']).agg(
    v1_sum=('value_1', 'sum'),
    v2_mean=('value_2', 'mean'),
    v3_var=('value_3', 'var'),
    n=('value_1', 'count'),
).reset_index()
print(f'agg_abc: {agg_abc.shape}')

In [ ]:
# Cell D: numpy linear algebra on a column.
# 20M-element correlation + percentile sweep -- pure numpy compute, no pandas.
v1 = df['value_1'].to_numpy()
v2 = df['value_2'].to_numpy()
v3 = df['value_3'].to_numpy()
v4 = df['value_4'].to_numpy()
corr_matrix = np.corrcoef(np.vstack([v1, v2, v3, v4]))
percentiles = np.percentile(v2, [1, 5, 25, 50, 75, 95, 99])
print(f'corr_matrix: {corr_matrix.shape}, percentiles: {percentiles}')

In [ ]:
# Cell E: derive a filtered subset and pivot it.
subset = df[df['group_c'] < 3]
pivot = subset.pivot_table(
    index='group_a', columns='group_b',
    values='value_1', aggfunc='mean',
)
print(f'subset: {subset.shape}, pivot: {pivot.shape}')

In [ ]:
# Cell F: summary using the cached aggregations from Cell B + Cell C.
# This cell is fast and mostly tests that downstream-of-heavy-cells
# benefits from cache hits on the warm path.
top_ab = agg_ab.nlargest(10, 'v2_sum')
top_abc = agg_abc.nlargest(10, 'n')
print(f'top_ab keys: {top_ab["group_a"].tolist()[:5]}')
print(f'top_abc keys: {top_abc["group_a"].tolist()[:5]}')